In [ ]:
!pip install -q transformers datasets peft accelerate evaluate

In [ ]:
# import torch
# from datasets import load_dataset
# from transformers import AutoImageProcessor, AutoModelForImageClassification

# # 1. Load CIFAR-10 and a pre-trained Vision Transformer
# dataset = load_dataset("cifar10", split="train[:5000]") # Tiny subset for fast checkpointing
# checkpoint = "google/vit-base-patch16-224-in21k"
# image_processor = AutoImageProcessor.from_pretrained(checkpoint)

# def transforms(examples):
#     examples["pixel_values"] = [image_processor(img.convert("RGB"), return_tensors="pt")["pixel_values"][0] for img in examples["img"]]
#     return examples

# dataset = dataset.with_transform(transforms)
# model = AutoModelForImageClassification.from_pretrained(checkpoint, num_labels=10)

In [ ]:
# from peft import LoraConfig, get_peft_model

# # Standard LoRA config, but with PiSSA initialization turned on!
# pissa_config = LoraConfig(
#     r=16,
#     lora_alpha=16,
#     target_modules=["query", "value"], # Targeting ViT attention layers
#     lora_dropout=0.0, # PiSSA requires dropout to be 0 for stability
#     init_lora_weights="pissa", # THIS is the NeurIPS paper implementation
# )

# # Apply PiSSA to the Vision Transformer
# pissa_model = get_peft_model(model, pissa_config)
# pissa_model.print_trainable_parameters()

In [ ]:
# from transformers import TrainingArguments, Trainer

# # Fast training arguments just to get the initial loss curve
# training_args = TrainingArguments(
#     output_dir="./pissa_vision_test",
#     remove_unused_columns=False,
#     eval_strategy="steps", # <-- THIS IS THE ONLY CHANGE
#     save_strategy="steps",
#     learning_rate=5e-4,
#     per_device_train_batch_size=32,
#     max_steps=50, # Just 50 steps for the Checkpoint 1 slide!
#     logging_steps=10,
# )

# trainer = Trainer(
#     model=pissa_model,
#     args=training_args,
#     train_dataset=dataset,
# )

# trainer.train()

In [ ]:
import os, random, numpy as np
import torch
from torch.utils.data import DataLoader
import torch.nn.functional as F

from datasets import load_dataset
from transformers import (
    ViTForImageClassification,
    ViTImageProcessor,
)
from peft import LoraConfig, get_peft_model

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def accuracy(logits, labels):
    preds = logits.argmax(dim=-1)
    return (preds == labels).float().mean().item()

def make_dataloaders(image_processor, train_size=5000, val_size=1000, batch_size=32):
    ds = load_dataset("cifar10")

    # small subsets for "overnight" runs
    train_ds = ds["train"].shuffle(seed=42).select(range(train_size))
    val_ds   = ds["test"].shuffle(seed=42).select(range(val_size))

    def collate_fn(batch):
        images = [x["img"] for x in batch]
        labels = torch.tensor([x["label"] for x in batch], dtype=torch.long)
        inputs = image_processor(images=images, return_tensors="pt")
        inputs["labels"] = labels
        return inputs

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    return train_loader, val_loader

def evaluate(model, val_loader, device):
    model.eval()
    accs = []
    with torch.no_grad():
        for batch in val_loader:
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)
            out = model(pixel_values=pixel_values)
            accs.append(accuracy(out.logits, labels))
    return float(np.mean(accs))

def train_one_epoch(model, train_loader, device, lr=2e-4, wd=0.01, log_every=100):
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    losses = []
    for step, batch in enumerate(train_loader, start=1):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        out = model(pixel_values=pixel_values, labels=labels)
        loss = out.loss

        loss.backward()
        opt.step()
        opt.zero_grad()

        losses.append(loss.item())
        if step % log_every == 0:
            print(f"step {step:4d} | loss {np.mean(losses[-log_every:]):.4f}")

    return float(np.mean(losses))

def build_vit_with_pissa_lora(model_name="google/vit-base-patch16-224-in21k",rank=8, alpha=16, dropout=0.05):
    """
    Applies LoRA to ViT attention projections. PiSSA is used only as an INIT method.
    """
    model = ViTForImageClassification.from_pretrained(model_name,num_labels=10,ignore_mismatched_sizes=True,  # replaces classifier head for CIFAR-10
    )
    # Typical targets in HF ViT: query/key/value are Linear layers inside attention.
    target_modules = ["query", "value"]  # keep it small & stable for first run

    # Try PiSSA init variants in descending order of "most likely to work fast"
    init_candidates = ["pissa_niter_4", "pissa", True]  # True = default LoRA init in PEFT
    last_err = None

    for init in init_candidates:
        try:
            lora_cfg = LoraConfig(r=rank,lora_alpha=alpha,lora_dropout=dropout,bias="none",
                                  target_modules=target_modules,task_type="SEQ_CLS",init_lora_weights=init,)
            peft_model = get_peft_model(model, lora_cfg)
            peft_model.print_trainable_parameters()
            # Show that adapters are inside real ViT attention modules
            hits = 0
            for name, _ in model.named_modules():
                if ("query" in name or "value" in name) and "lora_" in name:
                    print(name)
                    hits += 1
                    if hits >= 6:
                        break
            return peft_model, init
        except Exception as e:
            last_err = e
            continue

    raise RuntimeError(f"Failed to create PEFT model. Last error: {last_err}")

def main():
    set_seed(42)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device)

    image_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")
    train_loader, val_loader = make_dataloaders(
        image_processor=image_processor,
        train_size=5000,
        val_size=1000,
        batch_size=32 if device == "cuda" else 16
    )

    # --- Baseline (no LoRA) ---
    baseline = ViTForImageClassification.from_pretrained(
        "google/vit-base-patch16-224-in21k",
        num_labels=10,
        ignore_mismatched_sizes=True,
    ).to(device)

    # # Freeze everything except classifier head to get a quick baseline
    # for n, p in baseline.named_parameters():
    #     if "classifier" not in n:
    #         p.requires_grad = False

    print("\n[Baseline] full fine tuning")
    base_val0 = evaluate(baseline, val_loader, device)
    base_loss = train_one_epoch(baseline, train_loader, device, lr=2e-4, wd=0.01, log_every=200)
    base_val1 = evaluate(baseline, val_loader, device)
    print(f"Baseline val acc: {base_val0:.3f} -> {base_val1:.3f} | train loss ~ {base_loss:.4f}")

    # --- PiSSA LoRA ---
    print("\n[PiSSA-LoRA] (LoRA on attention projections)")
    pissa_model, init_used = build_vit_with_pissa_lora(rank=8, alpha=16, dropout=0.05)
    pissa_model = pissa_model.to(device)

    pissa_val0 = evaluate(pissa_model, val_loader, device)
    pissa_loss = train_one_epoch(pissa_model, train_loader, device, lr=2e-4, wd=0.01, log_every=200)
    pissa_val1 = evaluate(pissa_model, val_loader, device)
    print(f"PiSSA init used: {init_used}")
    print(f"PiSSA-LoRA val acc: {pissa_val0:.3f} -> {pissa_val1:.3f} | train loss ~ {pissa_loss:.4f}")

    print("\n=== Summary (copy into Slide 7) ===")
    print(f"Baseline (classifier-only) acc: {base_val0:.3f} -> {base_val1:.3f}")
    print(f"PiSSA-LoRA (attn q,v) acc:     {pissa_val0:.3f} -> {pissa_val1:.3f} (init={init_used})")

if __name__ == "__main__":
    main()

device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



[Baseline] full fine tuning
Baseline val acc: 0.068 -> 0.920 | train loss ~ 0.5868

[PiSSA-LoRA] (LoRA on attention projections)


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 302,602 || all params: 86,108,948 || trainable%: 0.3514
vit.encoder.layer.0.attention.attention.query.lora_dropout
vit.encoder.layer.0.attention.attention.query.lora_dropout.default
vit.encoder.layer.0.attention.attention.query.lora_A
vit.encoder.layer.0.attention.attention.query.lora_A.default
vit.encoder.layer.0.attention.attention.query.lora_B
vit.encoder.layer.0.attention.attention.query.lora_B.default
PiSSA init used: pissa_niter_4
PiSSA-LoRA val acc: 0.100 -> 0.963 | train loss ~ 1.5354

=== Summary (copy into Slide 7) ===
Baseline (classifier-only) acc: 0.068 -> 0.920
PiSSA-LoRA (attn q,v) acc:     0.100 -> 0.963 (init=pissa_niter_4)
